# Vintage CORE evaluation

This notebook installs the standalone repository and evaluates a pinned model on Original, Filtered, and Restyled CORE. Use a GPU runtime. The default is a ten-row-per-task smoke run; set `MAX_PER_TASK = -1` for the complete benchmark.

In [ ]:
MODEL_ID = "modern-d24"  # think-d12-r30, modern-d24, or gpt1900-d34
MAX_PER_TASK = 10            # -1 evaluates every row
SAVE_TO_DRIVE = True

In [ ]:
import os, subprocess, sys
from pathlib import Path

repo = Path("/content/vintage-core")
if repo.exists():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/johnny0595/vintage-core.git", str(repo)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{repo}[eval]"], check=True)

In [ ]:
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
except Exception:
    pass
print("HF_TOKEN available:", bool(os.environ.get("HF_TOKEN")))

In [ ]:
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    output_dir = Path("/content/drive/MyDrive/vintage-core-results") / MODEL_ID
else:
    output_dir = Path("/content/vintage-core-results") / MODEL_ID
output_dir.mkdir(parents=True, exist_ok=True)
command = [
    "vintage-core-eval",
    "--model", MODEL_ID,
    "--bundles", "original,filtered,restyled",
    "--max-per-task", str(MAX_PER_TASK),
    "--output-dir", str(output_dir),
]
subprocess.run(command, check=True, cwd=repo, env={**os.environ, "PYTHONUNBUFFERED": "1"})

In [ ]:
import pandas as pd
pd.read_csv(output_dir / "summary.csv")